# Interactive Talbot Carpet Simulation
### Near-Field Fresnel Diffraction & Wavefront Evolution

This interactive notebook models the spatial evolution of monochromatic light diffracted through a periodic transmission grating. Use the sliders below to vary experimental parameters in real time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from ipywidgets import interact, FloatSlider, IntSlider

In [ ]:
def plot_talbot(lpm=5.0, lam_nm=632.8, beam_width=2.0):
    # Physical parameters
    d = 1.0 / lpm                    # grating period (mm)
    lam = lam_nm * 1e-6              # wavelength (mm)
    Z_T = 2 * (d**2) / lam           # Talbot distance (mm)
    
    y_half = 1.8
    n_y = 600
    y = np.linspace(-y_half, y_half, n_y)
    
    z_max = 1.25 * Z_T
    n_z = 700
    z = np.linspace(1e-6 * Z_T, z_max, n_z)
    
    # Beam envelope and wave propagation
    k = 2 * np.pi / lam
    modes = np.arange(-35, 36)
    sigma = beam_width / (2 * np.sqrt(2 * np.log(2)))
    envelope = np.exp(-y**2 / (2 * sigma**2))
    
    E = np.zeros((n_y, n_z), dtype=complex)
    for n in modes:
        cn = 0.5 if n == 0 else np.sin(n * np.pi * 0.5) / (n * np.pi)
        kxn = 2 * np.pi * n / d
        kz_sq = k**2 - kxn**2
        if kz_sq <= 0:
            continue
        kzn = np.sqrt(kz_sq)
        E += cn * np.outer(np.exp(1j * kxn * y) * envelope, np.exp(1j * kzn * z))
        
    I = np.abs(E)**2
    I /= I.max()
    
    # Visualisation
    fig, ax = plt.subplots(figsize=(10, 4.5), dpi=120)
    im = ax.imshow(I, aspect='auto', origin='lower', extent=[0, z_max, y[0], y[-1]], cmap='hot')
    
    for frac, lbl in zip([0.25, 0.5, 0.75, 1.0], [r'$Z_T/4$', r'$Z_T/2$', r'$3Z_T/4$', r'$Z_T$']):
        ax.axvline(frac * Z_T, color='cyan', lw=0.8, ls='--', alpha=0.7)
        ax.text(frac * Z_T + 0.005 * z_max, y[-1] * 0.88, lbl, color='cyan', fontsize=9)
        
    ax.set_xlabel(r'Distance $z$ / $Z_T$')
    ax.set_ylabel('Position along sensor (mm)')
    ax.set_title(f'Talbot Carpet — Grating: {lpm} lp/mm | Wavelength: {lam_nm} nm | Z_T: {Z_T:.2f} mm')
    plt.show()

In [ ]:
interact(
    plot_talbot,
    lpm=FloatSlider(min=2.0, max=10.0, step=0.5, value=5.0, description='Grating lp/mm:'),
    lam_nm=FloatSlider(min=400.0, max=800.0, step=10.0, value=632.8, description='Lambda (nm):'),
    beam_width=FloatSlider(min=0.5, max=4.0, step=0.25, value=2.0, description='Beam FWHM (mm):')
);